In [29]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [31]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
191,This program would be useful for training hard...,negative
280,Filmmakers made a rather boring everyman's sto...,positive
647,Darr is a great movie! Shahrukh plays an obses...,positive
527,Here is another great film critics will love. ...,negative
372,Diagnosis Murder has been shown on most Weekda...,positive


In [32]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [33]:
df = normalize_text(df)
df.head()

,review,sentiment
191,program would useful training hardened felon b...,negative
280,filmmaker made rather boring everyman s story ...,positive
647,darr great movie shahrukh play obsessed lover ...,positive
527,another great film critic love problem good mo...,negative
372,diagnosis murder shown weekday afternoon bbc s...,positive


In [34]:
df['sentiment'].value_counts()

sentiment
positive    253
negative    247
Name: count, dtype: int64

In [35]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [36]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
191,program would useful training hardened felon b...,0
280,filmmaker made rather boring everyman s story ...,1
647,darr great movie shahrukh play obsessed lover ...,1
527,another great film critic love problem good mo...,0
372,diagnosis murder shown weekday afternoon bbc s...,1


In [24]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [37]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.50, random_state=42)

In [39]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Souritdev/Mlops-capstone-project.mlflow')
dagshub.init(repo_owner='Souritdev', repo_name='Mlops-capstone-project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression")


2025-11-19 21:20:17,192 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/Souritdev/Mlops-capstone-project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "Souritdev/Mlops-capstone-project"

2025-11-19 21:20:17,208 - INFO - Initialized MLflow to track repo "Souritdev/Mlops-capstone-project"


Repository Souritdev/Mlops-capstone-project initialized!

2025-11-19 21:20:17,211 - INFO - Repository Souritdev/Mlops-capstone-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/fc35069f60e74ad0aa2b2772a5e2518d', creation_time=1763564023106, experiment_id='5', last_update_time=1763564023106, lifecycle_stage='active', name='Logistic Regression', tags={}>

In [40]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib 

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 1000)
        mlflow.log_param("test_size", 0.50)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")

        # ---------------- NEW SAFE LOGGING METHOD ----------------
        os.makedirs("artifacts", exist_ok=True)
        joblib.dump(model, "artifacts/model.pkl")
        mlflow.log_artifact("artifacts/model.pkl", artifact_path="model")
        # ----------------------------------------------------------

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2025-11-19 21:20:22,576 - INFO - Starting MLflow run...
2025-11-19 21:20:23,043 - INFO - Logging preprocessing parameters...
2025-11-19 21:20:24,271 - INFO - Initializing Logistic Regression model...
2025-11-19 21:20:24,273 - INFO - Fitting the model...
2025-11-19 21:20:24,317 - INFO - Model training complete.
2025-11-19 21:20:24,318 - INFO - Logging model parameters...
2025-11-19 21:20:24,684 - INFO - Making predictions...
2025-11-19 21:20:24,687 - INFO - Calculating evaluation metrics...
2025-11-19 21:20:24,697 - INFO - Logging evaluation metrics...
2025-11-19 21:20:26,203 - INFO - Saving and logging the model...
2025-11-19 21:20:27,241 - INFO - Model training and logging completed in 4.20 seconds.
2025-11-19 21:20:27,242 - INFO - Accuracy: 0.644
2025-11-19 21:20:27,243 - INFO - Precision: 0.6744186046511628
2025-11-19 21:20:27,243 - INFO - Recall: 0.6492537313432836
2025-11-19 21:20:27,244 - INFO - F1 Score: 0.6615969581749049


🏃 View run clean-snipe-923 at: https://dagshub.com/Souritdev/Mlops-capstone-project.mlflow/#/experiments/5/runs/ad737758943f4f7b91e0cf5f78e09716
🧪 View experiment at: https://dagshub.com/Souritdev/Mlops-capstone-project.mlflow/#/experiments/5
